In [ ]:
# Import Section (importing all requiered libraries)
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

In [ ]:
train_path = "/Users/user/Documents/Sem2/Machine_Learning /Assignment2-1/TextBlob/examples/train.csv"
# defining the train path for reuseablity
# print("Using:", "/Users/user/Documents/Sem2/Machine_Learning /Assignment2-1/TextBlob/examples/train.csv")
df = pd.read_csv(train_path, encoding="cp1252")
# converting the csv into data frame for further training the model
print("Columns:", df.columns.tolist())
df = df.copy()

Using: /Users/user/Documents/Sem2/Machine_Learning /Assignment2-1/TextBlob/examples/train.csv
Columns: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)']


In [ ]:
text_col = "text"
label_col = "sentiment"
# Working with the above 2 columsn for building a svm model
print(df[[text_col, label_col]].head())
print("\nLabel distribution:")
print(df[label_col].value_counts())
# The above does the data preprocessing part
df = df[[text_col, label_col]].dropna().reset_index(drop=True)

                                                text sentiment
0                I`d have responded, if I were going   neutral
1      Sooo SAD I will miss you here in San Diego!!!  negative
2                          my boss is bullying me...  negative
3                     what interview! leave me alone  negative
4   Sons of ****, why couldn`t they put them on t...  negative

Label distribution:
sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64


In [ ]:
# Encoding labels (positive, negative, neutral) into numerical format
le = LabelEncoder()
y = le.fit_transform(df[label_col])
X = df[text_col]

print("Label classes:", le.classes_)

Label classes: ['negative' 'neutral' 'positive']


In [ ]:
# sploitting the data into train and test set in 80 : 20 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# defining the model pipeline
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=20000, ngram_range=(1, 2), stop_words="english"
            ),
        ),
        ("clf", LinearSVC(class_weight="balanced", random_state=42)),
    ]
)

In [ ]:
# Training the model on our data
print("Training baseline model...")
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_))

Training baseline model...
Accuracy: 0.673216885007278
              precision    recall  f1-score   support

    negative       0.66      0.66      0.66      1556
     neutral       0.64      0.64      0.64      2223
    positive       0.72      0.73      0.73      1717

    accuracy                           0.67      5496
   macro avg       0.68      0.68      0.68      5496
weighted avg       0.67      0.67      0.67      5496

Accuracy: 0.673216885007278
              precision    recall  f1-score   support

    negative       0.66      0.66      0.66      1556
     neutral       0.64      0.64      0.64      2223
    positive       0.72      0.73      0.73      1717

    accuracy                           0.67      5496
   macro avg       0.68      0.68      0.68      5496
weighted avg       0.67      0.67      0.67      5496



In [36]:
# simple test on the trained model
text = "Love you!"
predict = pipeline.predict([text])
predict
print(f'Sample prediction for "{text}":', le.inverse_transform(predict))

Sample prediction for "Love you!": ['positive']


In [ ]:
# Saving the trained model on our local folder for future resue
joblib.dump(
    pipeline,
    "/Users/user/Documents/Sem2/Machine_Learning /Assignment2-1/TextBlob/examples/sentiment_model.joblib",
)

['/Users/user/Documents/Sem2/Machine_Learning /Assignment2-1/TextBlob/examples/sentiment_model.joblib']